In [1]:
import fitz
import openai
from rapidfuzz import fuzz
from dotenv import load_dotenv

print("All libraries have been successfully loaded!")

ModuleNotFoundError: No module named 'fitz'

In [2]:
pip install groq


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: /Users/hhodzic/.local/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from groq import Groq

client = Groq(api_key="Your API")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [4]:
import fitz

def read_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    doc.close()
    return full_text

text = read_pdf("/Your Path")
print(text[:500])

Service Agreement
This Service Agreement (the "Agreement") is entered into as of January 1, 2025, between
DataFlow GmbH, a company incorporated under the laws of Austria, with its registered
office at Mariahilfer Strasse 100, 1060 Vienna, Austria ("Service Provider"), and Legal-
Corp AG, a company incorporated under the laws of Germany, with its registered office
at Hauptstrasse 45, 10115 Berlin, Germany ("Client").
1
1. Scope of Services
The Service Provider agrees to provide cloud-based data p


In [23]:
import re

def extract_clauses(text):
    # we look only for real clause numbers (1-20), not EUR amounts
    pattern = r'((?<!\d)([1-9]|1[0-9]|20)\.\s+[A-Z][^\n]+)'
    matches = list(re.finditer(pattern, text))
    
    clauses = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        clause_text = text[start:end].strip()
        clauses.append({
            "number": i+1,
            "title": match.group(0).strip(),
            "text": clause_text
        })
    
    return clauses

clauses = extract_clauses(text)

for c in clauses:
    print(f"Clause {c['number']}: {c['title']}")
    print(f"Length: {len(c['text'])} characters")
    print("---")

Clause 1: 1. Scope of Services
Length: 448 characters
---
Clause 2: 2. Payment Terms
Length: 411 characters
---
Clause 3: 3. Data Protection and Processing
Length: 502 characters
---
Clause 4: 4. Confidentiality
Length: 470 characters
---
Clause 5: 5. Intellectual Property
Length: 521 characters
---
Clause 6: 6. Limitation of Liability
Length: 455 characters
---
Clause 7: 7. Termination
Length: 415 characters
---
Clause 8: 8. Governing Law and Dispute Resolution
Length: 368 characters
---
Clause 9: 9. Automatic Renewal
Length: 336 characters
---
Clause 10: 10. Entire Agreement
Length: 577 characters
---


In [6]:
import json
from groq import Groq

client = Groq(api_key="YOUR API")

SYSTEM_PROMPT = """You are an expert contract lawyer. Analyze the given contract clause and identify any legal risks.

You must respond with a valid JSON object in exactly this format:
{
    "risk_level": "none" or "low" or "medium" or "high",
    "category": "short category name or null if no risk",
    "original_text": "exact quote of the risky part or null if no risk",
    "explanation": "short explanation of the risk or null if no risk",
    "gdpr_reference": "relevant GDPR article if applicable or null"
}

Risk categories to look for:
- Unilateral termination without notice
- Automatic renewal without notification
- Liability cap below contract value
- Unilateral fee increases
- Missing data breach notification (GDPR Article 33)
- Data transfer outside EEA without safeguards (GDPR Article 46)
- Unilateral contract amendment
- Excessive interest rates
- One-sided termination rights

Respond ONLY with the JSON object, no other text."""

def analyze_clause(clause_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Analyze this clause:\n\n{clause_text}"}
        ],
        temperature=0.1
    )
    
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)

# test only on the first clause
result = analyze_clause(clauses[0]["text"])
print(json.dumps(result, indent=2))

{
  "risk_level": "high",
  "category": "Unilateral termination without notice",
  "original_text": "The Service Provider reserves the right to modify, suspend, or discontinue any part of the services at any time without prior notice to the Client",
  "explanation": "The Service Provider can stop or change services without warning, potentially harming the Client's business operations",
  "gdpr_reference": null
}


In [7]:
print("Analyzing all clauses...\n")

results = []
for clause in clauses:
    print(f"Analyzing: {clause['title']}...")
    result = analyze_clause(clause["text"])
    result["clause_title"] = clause["title"]
    result["clause_number"] = clause["number"]
    results.append(result)

# show only risky ones
print("\n=== DETECTED RISKS ===\n")
for r in results:
    if r["risk_level"] != "none":
        print(f"[{r['risk_level'].upper()}] {r['clause_title']}")
        print(f"Category: {r['category']}")
        print(f"Text: {r['original_text']}")
        print(f"Explanation: {r['explanation']}")
        if r["gdpr_reference"]:
            print(f"GDPR: {r['gdpr_reference']}")
        print("---")

Analiziram sve klauzule...

Analiziram: 1. Scope of Services...
Analiziram: 2. Payment Terms...
Analiziram: 3. Data Protection and Processing...
Analiziram: 4. Confidentiality...
Analiziram: 5. Intellectual Property...
Analiziram: 6. Limitation of Liability...
Analiziram: 7. Termination...
Analiziram: 8. Governing Law and Dispute Resolution...
Analiziram: 9. Automatic Renewal...
Analiziram: 10. Entire Agreement...

=== PRONADJENI RIZICI ===

[HIGH] 1. Scope of Services
Kategorija: Unilateral termination without notice
Tekst: The Service Provider reserves the right to modify, suspend, or discontinue any part of the services at any time without prior notice to the Client
Objasnjenje: The Service Provider can stop or change services without warning, potentially harming the Client's business operations
---
[HIGH] 2. Payment Terms
Kategorija: Unilateral fee increases
Tekst: The Service Provider may increase fees at any time by providing written notice to the Client, with such increases taki

In [24]:
from rapidfuzz import fuzz, process

def find_position(original_text, search_text):
    # split the original into sentences
    sentences = original_text.split('.')
    
    best_match = None
    best_score = 0
    best_start = 0
    
    for sentence in sentences:
        score = fuzz.partial_ratio(search_text.lower(), sentence.lower())
        if score > best_score:
            best_score = score
            best_match = sentence.strip()
    
    # find position in the original text
    if best_match and best_score > 70:
        idx = original_text.lower().find(best_match.lower()[:30])
        return {
            "found": True,
            "score": best_score,
            "matched_text": best_match,
            "position": idx
        }
    
    return {"found": False, "score": best_score}

# test on the first risk
test_risk = results[0]["original_text"]
match = find_position(text, test_risk)
print(f"Searching: {test_risk[:80]}...")
print(f"Found: {match['found']}")
print(f"Score: {match['score']}")
print(f"Position in text: {match['position']}")
print(f"Matched: {match['matched_text'][:80]}...")

Searching: The Service Provider reserves the right to modify, suspend, or discontinue any p...
Found: True
Score: 99.31506849315068
Position in text: 721
Matched: The Service Provider reserves the right to modify, suspend, or discontinue
any p...


In [25]:
from IPython.display import HTML, display

def generate_html_report(original_text, results):
    highlighted_text = original_text
    
    # sort by risk level - high first
    risk_order = {"high": 0, "medium": 1, "low": 2, "none": 3}
    sorted_results = sorted(results, key=lambda x: risk_order.get(x["risk_level"], 3))
    
    colors = {
        "high": "#ffcccc",
        "medium": "#fff3cc",
        "low": "#cce5ff"
    }
    
    # highlight each risk in the text
    for r in sorted_results:
        if r["risk_level"] == "none" or not r.get("original_text"):
            continue
        
        risk_text = r["original_text"]
        color = colors.get(r["risk_level"], "#ffffff")
        tooltip = f"{r['category']}: {r['explanation']}"
        
        highlighted = f'<mark style="background-color: {color}; padding: 2px; border-radius: 3px;" title="{tooltip}">{risk_text}</mark>'
        highlighted_text = highlighted_text.replace(risk_text, highlighted)
    
    # create HTML report
    html = f"""
    <div style="font-family: Arial; max-width: 900px; margin: auto;">
        <h1 style="color: #333;">Contract Risk Analysis Report</h1>
        
        <div style="margin-bottom: 20px; padding: 15px; background: #f5f5f5; border-radius: 8px;">
            <h3>Legend</h3>
            <span style="background:#ffcccc; padding: 4px 10px; border-radius: 4px; margin-right: 10px;">High Risk</span>
            <span style="background:#fff3cc; padding: 4px 10px; border-radius: 4px; margin-right: 10px;">Medium Risk</span>
            <span style="background:#cce5ff; padding: 4px 10px; border-radius: 4px;">Low Risk</span>
        </div>
        
        <div style="margin-bottom: 30px; padding: 15px; background: #fff; border: 1px solid #ddd; border-radius: 8px;">
            <h3>Summary</h3>
            <p>Total risks found: <strong>{len([r for r in results if r['risk_level'] != 'none'])}</strong></p>
            <p>High: <strong style="color:red">{len([r for r in results if r['risk_level'] == 'high'])}</strong> &nbsp;
               Medium: <strong style="color:orange">{len([r for r in results if r['risk_level'] == 'medium'])}</strong> &nbsp;
               Low: <strong style="color:blue">{len([r for r in results if r['risk_level'] == 'low'])}</strong>
            </p>
        </div>
        
        <div style="padding: 20px; background: #fff; border: 1px solid #ddd; border-radius: 8px; white-space: pre-wrap; line-height: 1.8;">
            {highlighted_text}
        </div>
        
        <div style="margin-top: 30px;">
            <h3>Risk Details</h3>
    """
    
    for r in sorted_results:
        if r["risk_level"] == "none":
            continue
        color = colors.get(r["risk_level"], "#ffffff")
        gdpr = f"<br><strong>GDPR:</strong> {r['gdpr_reference']}" if r.get("gdpr_reference") else ""
        html += f"""
        <div style="margin-bottom: 15px; padding: 15px; background: {color}; border-radius: 8px;">
            <strong>[{r['risk_level'].upper()}] {r['clause_title']}</strong><br>
            <strong>Category:</strong> {r['category']}<br>
            <strong>Explanation:</strong> {r['explanation']}{gdpr}
        </div>
        """
    
    html += "</div></div>"
    return html

# generate and display the report
html_report = generate_html_report(text, results)
display(HTML(html_report))

In [26]:
# Conversational Q&A
conversation_history = []

def setup_conversation(original_text, results):
    # create context from the contract and the analysis results
    risk_summary = ""
    for r in results:
        if r["risk_level"] != "none":
            risk_summary += f"\n- [{r['risk_level'].upper()}] {r['clause_title']}: {r['explanation']}"
    
    system_context = f"""You are a legal assistant helping a lawyer review a contract.

Here is the full contract text:
---
{original_text}
---

Here is the risk analysis that was already performed:
---
{risk_summary}
---

Answer questions about the contract clearly and concisely. 
If asked to rewrite a clause, provide a safer alternative.
Always reference specific clause numbers when relevant."""
    
    return system_context

def chat(user_message, system_context):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_context},
            *conversation_history
        ],
        temperature=0.3
    )
    
    assistant_message = response.choices[0].message.content
    conversation_history.append({
        "role": "assistant", 
        "content": assistant_message
    })
    
    return assistant_message

# setup
system_context = setup_conversation(text, results)
print("Conversational Q&A ready. Ask a question!")

Conversational Q&A ready. Ask a question!


In [27]:
response = chat("Why is clause 6 high risk?", system_context)
print(response)

Clause 6 (Limitation of Liability) is considered high risk because it caps the Service Provider's total liability to the Client at EUR 500, which may be too low considering the potential value of the contract. This means that if the Service Provider causes significant damages to the Client, the Client's recourse would be limited to a maximum of EUR 500, regardless of the actual amount of damages incurred. This could leave the Client with insufficient compensation for potential losses, such as loss of data, loss of profits, or indirect damages, which are explicitly excluded from the Service Provider's liability (as stated in clause 6).


In [28]:
response = chat("If I sign this contract as the Client, what are the top 3 things I should negotiate before signing?", system_context)
print(response)

Based on the risk analysis, the top 3 things you should negotiate before signing the contract as the Client are:

1. **Clause 6 (Limitation of Liability)**: Negotiate a higher liability cap or remove the cap altogether to ensure you have sufficient recourse in case of significant damages. Consider a more reasonable cap, such as a percentage of the annual contract value.
2. **Clause 2 (Payment Terms)**: Negotiate a notice period for fee increases and consider capping the amount of fee increases within a certain timeframe. This would prevent unexpected and significant cost increases.
3. **Clause 7 (Termination)**: Negotiate more balanced termination rights, such as allowing the Client to terminate the agreement with a reasonable notice period (e.g., 30 or 60 days) without requiring a material breach by the Service Provider. This would provide more flexibility and protection for the Client.

These three clauses pose significant risks to the Client, and negotiating them could help mitigate

In [29]:
response = chat("Rewrite clause 7 to be more balanced for both parties.", system_context)
print(response)

Here's a rewritten version of clause 7 (Termination):

7. Termination
Either party may terminate this Agreement upon written notice to the other party, with a notice period of 30 days for termination without cause and 90 days for termination due to a material breach. In the event of a material breach, the non-breaching party must provide written notice to the breaching party, specifying the breach and allowing the breaching party 30 days to cure the breach. If the breaching party fails to cure the breach within the specified timeframe, the non-breaching party may terminate the Agreement.

This rewritten clause provides more balance by:

* Allowing either party to terminate the agreement with a reasonable notice period (30 days for termination without cause)
* Requiring a notice period for termination due to a material breach (90 days)
* Providing an opportunity for the breaching party to cure the breach within a specified timeframe (30 days)
* Allowing the non-breaching party to termin

In [30]:
GDPR_DORA_CONTEXT = """
GDPR KEY ARTICLES:

Article 28 - Processor obligations:
A processor shall not engage another processor without prior specific or general written 
authorisation of the controller. The processor shall implement appropriate technical and 
organisational measures to ensure a level of security appropriate to the risk. The processor 
shall assist the controller in ensuring compliance with data breach notification obligations.

Article 33 - Notification of personal data breach:
In the case of a personal data breach, the controller shall notify the supervisory authority 
without undue delay and, where feasible, not later than 72 hours after having become aware 
of it. The processor must notify the controller without undue delay after becoming aware of 
a personal data breach.

Article 46 - Transfers subject to appropriate safeguards:
A controller or processor may transfer personal data to a third country only if the controller 
or processor has provided appropriate safeguards, and on condition that enforceable data 
subject rights and effective legal remedies for data subjects are available.

Article 82 - Right to compensation:
Any person who has suffered material or non-material damage as a result of an infringement 
of this Regulation shall have the right to receive compensation from the controller or processor.

---

DORA KEY ARTICLES (Digital Operational Resilience Act):

Article 28 - ICT third-party risk:
Financial entities shall only enter into contractual arrangements with ICT third-party service 
providers that comply with appropriate information security standards. Contracts must include 
full service level descriptions, data locations, provisions on availability, integrity and 
security of data, and termination rights.

Article 30 - Key contractual provisions:
Contracts with ICT providers must include: clear description of all functions and services, 
locations where data is processed, provisions on availability and data integrity, ICT security 
requirements, audit rights for the financial entity, termination rights in case of breach, 
exit strategies, and mandatory incident reporting obligations.

Article 19 - Reporting of major ICT incidents:
Financial entities shall report major ICT-related incidents to the competent authority. 
ICT third-party service providers must notify and support the financial entity in reporting 
obligations without undue delay.
"""

def setup_conversation_with_regulations(original_text, results):
    risk_summary = ""
    for r in results:
        if r["risk_level"] != "none":
            risk_summary += f"\n- [{r['risk_level'].upper()}] {r['clause_title']}: {r['explanation']}"
    
    system_context = f"""You are a legal assistant helping a lawyer review a contract.
You have access to the following regulatory frameworks:

{GDPR_DORA_CONTEXT}

Here is the full contract text:
---
{original_text}
---

Here is the risk analysis already performed:
---
{risk_summary}
---

When answering questions, always reference specific GDPR or DORA articles where relevant.
Be precise and cite exact article numbers."""
    
    return system_context


conversation_history = []
system_context = setup_conversation_with_regulations(text, results)
print("Context with GDPR and DORA ready!")

Context with GDPR and DORA ready!


In [31]:
response = chat("Does clause 3 comply with GDPR and DORA requirements?", system_context)
print(response)

Clause 3 of the contract does not fully comply with GDPR and DORA requirements. 

According to GDPR Article 28, a processor (in this case, the Service Provider) shall implement appropriate technical and organisational measures to ensure a level of security appropriate to the risk. Clause 3 states that the Service Provider will implement such measures, which aligns with GDPR Article 28.

However, Clause 3 also states that the Service Provider shall not be required to notify the Client in the event of a personal data breach. This contradicts GDPR Article 33, which requires the processor to notify the controller (in this case, the Client) without undue delay after becoming aware of a personal data breach.

Additionally, DORA Article 30 requires contracts with ICT providers to include provisions on availability, integrity, and security of data. While Clause 3 mentions that the Service Provider will implement technical and organisational measures, it does not explicitly address availability

In [32]:
def get_clause_by_number(clause_number: int):
    for c in clauses:
        if c["number"] == clause_number:
            return c["text"]
    return "Clause not found"

def get_risk_for_clause(clause_number: int):
    for r in results:
        if r["clause_number"] == clause_number:
            return r
    return {"explanation": "No risk data found"}

def rewrite_clause(clause_number: int):
    clause_text = get_clause_by_number(clause_number)
    risk = get_risk_for_clause(clause_number)
    print(f"DEBUG - clause_text: {clause_text[:100]}")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a legal expert. Rewrite the following clause to be safer and more balanced for both parties. Always include the rewritten clause in your response."},
            {"role": "user", "content": f"Here is clause {clause_number} that needs to be rewritten:\n\n{clause_text}\n\nRisk identified: {risk.get('explanation', 'General risk')}\n\nPlease provide a safer rewrite of this exact clause."}
        ],
        temperature=0.3
    )
    result = response.choices[0].message.content
    print(f"DEBUG - tool result: {result[:100]}")
    return result

def get_regulatory_analysis(clause_number: int):
    clause_text = get_clause_by_number(clause_number)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": f"You are a legal expert. Analyze the clause against GDPR and DORA.\n\n{GDPR_DORA_CONTEXT}"},
            {"role": "user", "content": f"Analyze this clause for GDPR and DORA compliance:\n\n{clause_text}"}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

def summarize_all_risks():
    high = [r for r in results if r["risk_level"] == "high"]
    medium = [r for r in results if r["risk_level"] == "medium"]
    low = [r for r in results if r["risk_level"] == "low"]
    summary = f"Total risks: {len(high) + len(medium) + len(low)}\n"
    summary += f"High: {len(high)}, Medium: {len(medium)}, Low: {len(low)}\n\n"
    for r in high:
        summary += f"[HIGH] {r['clause_title']}: {r['explanation']}\n"
    return summary

print("Tools defined!")

Tools defined!


In [17]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_clause_by_number",
            "description": "Get the full text of a specific clause by its number",
            "parameters": {
                "type": "object",
                "properties": {
                    "clause_number": {
                        "type": "integer",
                        "description": "The clause number (1-10)"
                    }
                },
                "required": ["clause_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rewrite_clause",
            "description": "Rewrite a risky clause to be safer and more balanced for both parties",
            "parameters": {
                "type": "object",
                "properties": {
                    "clause_number": {
                        "type": "integer",
                        "description": "The clause number to rewrite"
                    }
                },
                "required": ["clause_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_regulatory_analysis",
            "description": "Analyze a clause for GDPR and DORA compliance with specific article references",
            "parameters": {
                "type": "object",
                "properties": {
                    "clause_number": {
                        "type": "integer",
                        "description": "The clause number to analyze"
                    }
                },
                "required": ["clause_number"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "summarize_all_risks",
            "description": "Get a complete summary of all risks found in the contract",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

print("Tools LLM!")

Tools opisani za LLM!


In [33]:
import json

tool_map = {
    "get_clause_by_number": get_clause_by_number,
    "rewrite_clause": rewrite_clause,
    "get_regulatory_analysis": get_regulatory_analysis,
    "summarize_all_risks": summarize_all_risks
}

def agent(user_message):
    print(f"\nUser: {user_message}")
    
    messages = [
        {"role": "system", "content": f"You are a legal assistant for contract review. Use the available tools to answer questions about the contract.\n\n{GDPR_DORA_CONTEXT}"},
        {"role": "user", "content": user_message}
    ]
    
    # first call - the LLM decides which tool to call
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.1
    )
    
    message = response.choices[0].message
    
    # if the LLM decides to call a tool
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        
        print(f"Agent calls tool: {tool_name} with arguments: {tool_args}")
        
        # call the actual tool
        tool_result = tool_map[tool_name](**tool_args)
        
        # add the tool result to the history
        messages.append(message)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })
        
        # second call - the LLM formulates the final answer
        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            temperature=0.3
        )
        
        return final_response.choices[0].message.content
    
    # if the LLM responds directly without a tool
    return message.content

print("Agent ready!")

Agent ready!


In [19]:
print(agent("Rewrite clause 7 to be more balanced"))


Korisnik: Rewrite clause 7 to be more balanced
Agent poziva tool: rewrite_clause s argumentima: {'clause_number': 7}
DEBUG - clause_text: 7. Termination
The Service Provider may terminate this Agreement at any time and for any reason by
p
DEBUG - tool result: To address the identified risk and create a more balanced contract, the termination clause can be re
7. Termination
Either party may terminate this Agreement upon written notice to the other party, provided that the terminating party has first given the other party at least 30 days' written notice of its intention to terminate. However, if the terminating party is the Client, such termination must be based on a material breach by the Service Provider of its obligations under this Agreement, which breach has not been remedied within 30 days of written notice from the Client. Similarly, if the terminating party is the Service Provider, such termination must be based on a material breach by the Client of its obligations under this A

In [34]:
print(get_clause_by_number(7))

7. Termination
The Service Provider may terminate this Agreement at any time and for any reason by
providing written notice to the Client. The Client may only terminate this Agreement if
the Service Provider has materially breached its obligations and has failed to remedy such
breach within 90 days of written notice. Upon termination by either party, all outstanding
invoices become immediately due and payable.
8


In [21]:
print(agent("Does clause 3 violate any GDPR articles?"))


Korisnik: Does clause 3 violate any GDPR articles?
Agent poziva tool: get_regulatory_analysis s argumentima: {'clause_number': 3}
Based on the analysis, clause 3 may violate GDPR Articles 28, 33, 46, and 82, as well as DORA Articles 28, 30, and 19. To ensure compliance, it is recommended to revise the clause to include explicit language on the Service Provider's obligations to notify the Client in the event of a personal data breach, specify appropriate safeguards for transfers of personal data, include key contractual provisions, address the reporting of major ICT-related incidents, and consider adding language on the right to compensation for data subjects.


In [22]:
print(agent("Give me a summary of all risks in this contract"))


Korisnik: Give me a summary of all risks in this contract
Agent poziva tool: summarize_all_risks s argumentima: {}
Based on the provided contract, the following risks have been identified:

1. **Data Breach Risk**: The contract does not explicitly require the Service Provider to notify the Client in case of a data breach, which is a violation of GDPR Article 33.
2. **ICT Third-Party Risk**: The contract does not ensure that the Service Provider complies with appropriate information security standards, as required by DORA Article 28.
3. **Contractual Provisions Risk**: The contract may not include key contractual provisions, such as clear descriptions of functions and services, data locations, provisions on availability and data integrity, ICT security requirements, audit rights, termination rights, exit strategies, and incident reporting obligations, as required by DORA Article 30.
4. **Reporting of Major ICT Incidents Risk**: The contract may not require the Service Provider to notif